# C5 · 캡스톤 5: 확률적 SIR 전염병 모형 (Capstone 5: Stochastic SIR Epidemics — CTMC, Branching Approximation and Final Size)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §4.1 (CTMC 정의), §4.2 (전이확률 계산), §4.4 (탈출 분포·탈출 시간), §1.10 (분기 과정) · Ross 12e §5.2–5.3 (지수 경쟁·푸아송), Ch. 6 (CTMC·출생–사망), Ch. 4 (분기 과정 절) · Lawler 2e §2.4 (분기 과정), Ch. 3 (CTMC) · Allen, *Stochastic Processes with Applications to Biology* 2e, CTMC 생물학 응용 장 (SIS/SIR CTMC) · Andersson & Britton (2000) Ch. 2–4; Britton (2010) *Math. Biosci.* 225 §2–3 (마르코프 SIR, 분기 근사, 최종 규모) · Barbour (1975) *Biometrika* 62; Ball & Donnelly (1995) *SPA* 55 (시간 이동·커플링) |
| 재사용 노트북 | 03a (Gillespie·생성원), 01g (골턴–왓슨·`extinction_probability`·`galton_watson`), 02a (경쟁하는 지수 시계), 03b (`transition_matrix` = expm), 01c (흡수확률·기본행렬), 05d (마팅게일 극한 $W$), 00b (SE), C4 (Recall만) |
| 예상 소요 | 150분 |
| 상태 | draft |

## 프로젝트 브리프 — 합격 기준과 단계

**브리프.** 마르코프 SIR 전염병 모형(인구 $N = 1000$, 접촉률 $\beta = 2$, 회복률 $\gamma = 1$, $R_0 = \beta/\gamma = 2$, 초기 감염자 $I_0 = 1$; 시간 단위 = 평균 감염기간 $1/\gamma$)을 상태 $(S, I)$ 위의 CTMC로 정의하고 Gillespie로 시뮬레이션하여 결정론적 ODE와 대조한다. 핵심 질문은 네 가지다. (i) 감염자 1명에서 시작하면 얼마나 자주 초기에 사그라지는가 — 분기 과정 근사는 $1/R_0$, 정확값은 최종 규모 DP로. (ii) 대유행이 나면 최종 규모는 얼마인가 — 최종 규모 방정식 $z = 1 - e^{-R_0 z}$와 이봉(bimodal) 분포. (iii) 정점 시각은 왜, 얼마나 흩어지는가 — 초기 단계의 마팅게일 극한 $W$(05d)에 의한 시간 이동. (iv) 작은 $N$에서는 생성원 행렬 하나로 이 모든 것을 정확히 계산할 수 있는가(03a/03b/01c).

**합격 기준 — 이 그림·숫자가 나오면 성공.**

1. $N = 30$: 496상태 생성원 $Q$에서 `transition_matrix`가 준 흡수확률(최종 규모 pmf)이 점프 사슬 DP와 $10^{-10}$ 이내 일치; $P(I_5 = 0) = 0.66260$, $E[T_{ext}] = 3.3408$이 Gillespie와 4 SE 이내.
2. $N = 1000$: 최종 규모 히스토그램이 이봉(0.502가 $Z \le 50$, 나머지가 800 근처)이고 정확 pmf(DP)와 겹침; $P(Z \le 100)$ MC가 정확값 0.50207과 4 SE 이내(분기 근사 0.5와의 차이 0.002는 $O(1/N)$); $E[Z \mid \text{major}]$가 795.375와 4 SE 이내(ODE 최종 규모 797).
3. 기하 자손분포($p = 1/3$)의 `extinction_probability` = 0.5; 10세대 소멸확률 0.49976이 `galton_watson`과 4 SE 이내; $I_0 = 3$에서 $P(\text{minor}) = 0.12774 \approx (1/2)^3$.
4. 대유행 조건부 정점 시각의 히스토그램(평균 ≈ 6.63, sd ≈ 1.45)이 ODE 정점 6.757 주위에 퍼져 있고 시간 이동 법칙 밀도(평균 6.64, sd 1.28)와 대체로 겹침.
5. $R_0 \in \{1.5, 2, 3\}$에서 정확 $P(\text{minor})$(0.6732, 0.5021, 0.3341)가 $1/R_0$과, $E[Z \mid \text{major}]/N$(0.578, 0.795, 0.940)이 $z(R_0)$(0.583, 0.797, 0.940)와 나란히 그려짐.

**단계.**

| 단계 | 내용 | 도구 |
|---|---|---|
| 1 | 작은 $N = 30$: 생성원, expm, 흡수확률, 기대 소멸시간, Gillespie(단일 경로 + 벡터화) | `is_generator`, `transition_matrix`, `embedded_chain`, `gillespie`, inline `sir_generator`, `sir_final_size_pmf`, `sir_gillespie_batch` |
| 2 | $N = 1000$ 벡터화 Gillespie와 최종 규모의 이봉 분포, ODE 대조 | inline `sir_gillespie_batch`(단계 1), `sir_gillespie_path`, `sir_ode` |
| 3 | 분기 과정 근사: 기하 자손, $1/R_0$, 총 자손 분포 | `extinction_probability`, `galton_watson` |
| 4 | 정점 시각 분포와 시간 이동 법칙 | inline `peak_time_shift_density` |
| 5 | $R_0$ 민감도: 소멸확률과 최종 규모 | DP + MC |
| Compare | 모든 검증 수치를 한 표로 | `compare_table`, `assert_close` |
| 6 | 전염병 보고서 | — |
| 스트레치 | S1 $R_0$ 추정, S2 백신·집단면역 문턱, S3 $P(\text{minor}) - 1/R_0$의 $N$-스케일링 | — |

## 0. 준비 (Setup)

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
from scipy.special import comb
from scipy.stats import ks_1samp
from IPython.display import Markdown
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.ctmc import is_generator, transition_matrix, embedded_chain, gillespie
from spkit.markov import extinction_probability, galton_watson
from spkit.plots import plot_hist_vs_pdf

SEED, rng = rng_for("C5")
setup_plots()
T_START = time.perf_counter()
N, BETA, GAMMA, I0 = 1000, 2.0, 1.0, 1        # 기본 파라미터 (시간 단위 = 1/gamma)
R0 = BETA / GAMMA
MINOR = 100                                    # 소규모 유행 문턱: Z <= 100
N_PATHS = scale(20_000)                        # N=1000 Gillespie 경로 수 (FAST: 2,000)
N_SMALL = scale(200_000)                       # N=30 벡터화 Gillespie 경로 수 (FAST: 20,000)
R0_MC = [1.5] if fast() else [1.5, 3.0]        # 단계 5의 MC 대상
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, N_SMALL={N_SMALL}")

## 1. Recall — 재사용 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (C4) 대조 변량의 최적 계수와 그때의 분산 감소는?

<details><summary>정답</summary>

$b^* = \mathrm{Cov}(Y, C)/\mathrm{Var}(C)$, $\mathrm{Var}(Y_{b^*}) = \mathrm{Var}(Y)(1 - \rho_{YC}^2)$. 기하 아시안을 대조 변량으로 쓰면 $\rho = 0.9996$ → 분산 약 1,300배 감소.

</details>

**Q2.** (C4) 이산 관측 장벽 옵션 MC의 편향은 어떤 차수이며 어떻게 없애는가?

<details><summary>정답</summary>

$O(\sqrt{\Delta t})$. BGK 장벽 이동 $He^{\pm 0.5826\sigma\sqrt{\Delta t}}$로 근사 보정하거나, 브라운 다리 교차확률 $\exp(-2(a - h)(b - h)/(\sigma^2\Delta t))$로 무편향화한다. 경로 수를 늘려도 줄지 않는다(분산이 아니라 편향).

</details>

**Q3.** (C4) 경로별 그릭스 추정량이 유효한 조건과 디지털 옵션에서 실패하는 이유는?

<details><summary>정답</summary>

페이오프가 파라미터에 대해 립시츠(미분과 기대의 교환 가능). 디지털은 계단함수라 경로별 도함수가 a.s. 0 → 우도비 추정량을 쓴다.

</details>

**Q4.** (07e, 두 노트북 전) 위험중립 측도에서 GBM의 드리프트는? 그 아래 무엇이 마팅게일인가?

<details><summary>정답</summary>

$r$: $dS = rS\,dt + \sigma S\,dW^*$. 할인 가격 $e^{-rt}S_t$가 마팅게일이라 $V_0 = E^*[e^{-rT}h]$. (이 캡스톤의 마팅게일은 $e^{-(\beta - \gamma)t}I_t$ — 정리 4.)

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **마르코프 SIR CTMC.** 상태 $(S, I)$, $R = N - S - I$. 전이 $(S, I) \to (S - 1, I + 1)$ 율 $\beta SI/N$(감염), $(S, I) \to (S, I - 1)$ 율 $\gamma I$(회복). $I = 0$이면 흡수. $R_0 = \beta/\gamma$. 기본 파라미터 $N = 1000$, $\beta = 2$, $\gamma = 1$, $I_0 = 1$.
- **Gillespie (03a).** 총 율 $a = \beta SI/N + \gamma I$, 체류시간 $\sim$ Exp($a$), 사건은 감염 w.p. $(\beta SI/N)/a$(02a 경쟁하는 시계). 벡터화: 경로 축으로 배열 $(S, I, t)$를 동시에 갱신하고 $I = 0$인 경로는 비활성화한다.
- **최종 규모.** $Z$ = 유행 전체에서 감염된 총 인원(초기 감염자 포함) $= N - S_\infty$. 소규모(minor) 유행 $Z \le 100$, 대유행(major) $Z > 100$. 정확 pmf에서 $Z \in [51, 650]$의 질량이 $10^{-4}$ 미만이므로 문턱은 둔감하다.
- **결정론적 SIR ODE.** $\dot S = -\beta SI/N$, $\dot I = \beta SI/N - \gamma I$, $S(0) = N - 1$, $I(0) = 1$. 최종 규모 방정식 $z = 1 - e^{-R_0 z}$($z = Z/N$ 극한, $S_0 \approx N$). 정점은 $S = N/R_0$일 때이고 $I_{max}/N = 1 - 1/R_0 - \ln R_0/R_0$.
- **분기 과정 근사.** $S \approx N$인 초기 단계에서 감염자 한 명이 만드는 2차 감염 수 $K \sim$ Geometric on $\{0, 1, \dots\}$, $P(K = k) = (1 - p)^k p$, $p = \gamma/(\beta + \gamma)$($R_0 = 2$: $p = 1/3$), $E[K] = R_0$, pgf $f(s) = p/(1 - (1 - p)s)$. 세대별 소멸확률 $q_n = f^{\circ n}(0)$.
- **마팅게일 극한 (05d의 연속시간판).** 선형 출생–사망 과정(출생 $\beta$, 사망 $\gamma$, 1명 시작, $\beta > \gamma$ 즉 $R_0 > 1$일 때)에서 $e^{-(\beta - \gamma)t}I_t \to W$ a.s.; $P(W = 0) = \gamma/\beta = 1/R_0$, $W \mid W > 0 \sim$ Exp(평균 $\theta = \beta/(\beta - \gamma)$)($R_0 = 2$: $\theta = 2$). 초기 성장 $I_t \approx We^{(\beta - \gamma)t}$.

### 2.2 정리 1 — 점프 사슬과 최종 규모의 DP (Durrett 3e §4.1, §4.4; Ross 12e §5.2)

> 상태 $(S, I)$, $I > 0$에서 다음 사건이 감염일 확률은
> $$p(S) = \frac{\beta SI/N}{\beta SI/N + \gamma I} = \frac{R_0 S}{R_0 S + N}$$
> 로 $I$에 무관하다. 따라서 최종 규모 분포는 점프 사슬만으로 정해지며, 상태 방문 확률 $f(S, I)$를 $f(S - 1, I + 1) \mathrel{+}= f(S, I)\,p(S)$, $f(S, I - 1) \mathrel{+}= f(S, I)(1 - p(S))$로 $S$ 내림차순·$I$ 내림차순으로 전파하면 $P(S_\infty = s) = f(s, 0)$ ($O(N^2)$, $N = 1000$에 0.5초).

**가정이 왜 필요한가**
- **경쟁하는 지수 시계 (02a)**: 두 독립 지수 시계 중 첫 번째가 감염일 확률 $= a/(a + b)$ — 마르코프(지수) 감염기간이 필요하다. 고정 감염기간이면 사건 순서가 마르코프가 아니다(E3).
- **$I$가 약분되는 것**은 두 율 모두 $I$에 비례하기 때문(질량작용 접촉률 $\beta SI/N$). 접촉률이 $\beta SI$(밀도 의존)여도 같지만, 포화 접촉률이면 $p$가 $I$에 의존해 DP가 2차원 격자 전체를 그대로 요구한다(그래도 가능하지만 $I$-무관성은 사라진다).

<details><summary>증명</summary>

감염 시계 $\sim$ Exp($\beta SI/N$), 회복 시계 $\sim$ Exp($\gamma I$)가 독립이므로 02a의 경쟁하는 시계 정리에 의해 $P(\text{감염 먼저}) = \frac{\beta SI/N}{\beta SI/N + \gamma I}$이고 분자·분모의 $I$가 약분되어 $\frac{\beta S/N}{\beta S/N + \gamma} = \frac{R_0 S}{R_0 S + N}$. 사건 유형의 열은 $(S, I)$ 위의 마르코프 사슬(점프 사슬, 03a `embedded_chain`)이고, 최종 규모는 흡수 상태 $(s, 0)$의 함수이므로 체류시간과 무관하다 — 흡수확률(01c)만 필요하다. 격자 $(S, I)$ 위에서 상태 $(S, I)$로 들어오는 전이는 $(S + 1, I - 1)$(감염)과 $(S, I + 1)$(회복)에서만 오므로, $S$ 내림차순·같은 $S$ 안에서는 $I$ 내림차순으로 처리하면 $f(S, I)$에 들어올 질량이 모두 도착한 뒤에 밖으로 내보낼 수 있다. 이것이 흡수확률의 전방 전파(forward propagation)이며 $\sum_s f(s, 0) = 1$이다. $\square$

</details>

### 2.3 정리 2 — 분기 과정 근사와 소멸확률 $1/R_0$ (Durrett 3e §1.10; Lawler 2e §2.4)

> 초기 단계($S \approx N$)에서 감염자 한 명의 2차 감염 수는 $K \sim \mathrm{Geom}(p)$, $p = \gamma/(\beta + \gamma)$, $E[K] = \beta/\gamma = R_0$. 대응하는 골턴–왓슨 과정의 소멸확률은 $q = \min(1, 1/R_0)$; $I_0 = k$명이면 $q^k$. $R_0 = 2$: $q = 1/2$, $I_0 = 3$: $1/8$ (정확 CTMC: 0.50207, 0.12774).

**가정이 왜 필요한가**
- **$S \approx N$**: 각 접촉이 감수성자와 이루어진다는 가정. 감수성자가 줄면 유효 $R_0$가 떨어져 정확 소멸확률이 $1/R_0$보다 약간 크다($N(P - 1/R_0) \approx 2.0$, S3).
- **감염자 간 독립**: 분기 과정의 핵심. 유한 $N$에서는 감수성 풀을 공유해 약한 종속이 생긴다.
- **지수 감염기간**: 기하 자손분포의 근거. 고정 감염기간이면 $K \sim$ Poisson($R_0$)이고 소멸확률은 $s = e^{R_0(s - 1)}$의 해 0.20319 (E2/E3) — $1/R_0$은 마르코프 모형 특유의 값이다.

<details><summary>증명</summary>

감염자 한 명은 율 $\beta$로 감염 사건, 율 $\gamma$로 회복 사건을 겪는다(독립 지수 시계, $S \approx N$). 회복 전까지 사건이 반복되는 동안 매 사건이 감염일 확률은 $\beta/(\beta + \gamma) = 1 - p$이고 회복이면 종료하므로, 회복 전 감염 횟수는 $K \sim \mathrm{Geom}(p)$ on $\{0, 1, \dots\}$: $P(K = k) = (1 - p)^k p$, $E[K] = (1 - p)/p = \beta/\gamma = R_0$. pgf는 $f(s) = \sum_k (1 - p)^k p\, s^k = p/(1 - (1 - p)s)$. 고정점 $s = p/(1 - (1 - p)s)$ ⇔ $(1 - p)s^2 - s + p = 0$ ⇔ $s \in \{1,\ p/(1 - p)\}$, 그리고 $p/(1 - p) = \gamma/\beta = 1/R_0$. 01g의 정리(소멸확률은 $[0, 1]$에서 가장 작은 고정점)에 의해 $R_0 > 1$이면 $q = 1/R_0$, $R_0 \le 1$이면 $q = 1$. 독립 계통 $k$개가 모두 소멸할 확률은 $q^k$. $\square$

</details>

### 2.4 정리 3 — 결정론적 극한: 최종 규모 방정식과 정점 (Andersson & Britton Ch. 2–4; Britton 2010 §2)

> $N \to \infty$에서 $(S_t/N, I_t/N)$은 SIR ODE의 해로 수렴한다(Kurtz의 밀도 의존 CTMC 대수의 법칙; Ethier–Kurtz Ch. 11 인용). ODE에서 $S_\infty = S_0\exp(-R_0(N - S_\infty)/N)$; $S_0 \approx N$이면 $z = 1 - e^{-R_0 z}$. 정점($\dot I = 0$)은 $S = N/R_0$에서이고 $I_{max}/N = 1 - 1/R_0 - \ln R_0/R_0$ ($S_0 \approx N$, $I_0 \approx 0$).

**가정이 왜 필요한가**
- **$N \to \infty$와 $I_0/N \to i_0 > 0$**: $I_0 = 1$ 고정이면 초기 단계는 분기 과정(확률적)이고 ODE는 대유행 경로의 '모양'만 맞춘다 — 그래서 시간 이동 법칙(정리 4)이 필요하다.
- **$S_0 \approx N$**: 백신 접종 등으로 $S_0 = N(1 - v)$이면 방정식이 $z = (1 - v)(1 - e^{-R_0 z})$로 바뀐다(S2).

**증명 스케치.** $dS/dR = (-\beta SI/N)/(\gamma I) = -R_0 S/N$ → $S(t) = S_0 e^{-R_0 R(t)/N}$. $t \to \infty$에서 $I_\infty = 0$, $R_\infty = N - S_\infty$를 넣으면 최종 규모 방정식. 정점: $dI/dS = -1 + N/(R_0 S)$를 $S_0 = N$에서 $N/R_0$까지 적분하면 $I_{max} = N - N/R_0 - (N/R_0)\ln R_0$. 수치: $z(2) = 0.796812$; ODE($S_0 = 999$, $I_0 = 1$) 최종 규모 0.797154, 정점 153.93(점근 공식 153.43), 정점 시각 6.7573.

### 2.5 정리 4 — 정점 시각의 시간 이동 법칙 (Barbour 1975; Ball & Donnelly 1995)

> 대유행 조건부로, 확률적 유행 곡선은 ODE($I(0) = 1$) 곡선을 시간축에서 $-\ln W/(\beta - \gamma)$만큼 옮긴 것에 가깝다:
> $$T_{peak} \approx T^{ODE}_{peak} - \frac{\ln W}{\beta - \gamma}, \qquad W \sim \mathrm{Exp}\Big(\text{평균 } \theta = \frac{\beta}{\beta - \gamma}\Big).$$
> 따라서 $E[T_{peak}] \approx T^{ODE} - (\ln\theta - \gamma_E)/(\beta - \gamma)$, $\mathrm{sd} \approx \pi/(\sqrt 6(\beta - \gamma))$, 중앙값 $T^{ODE} - \ln(\theta\ln 2)/(\beta - \gamma)$. $R_0 = 2$: 평균 6.641, sd 1.283, 중앙값 6.431 (MC, 20,000경로: 6.63, 1.45, 6.43).

**가정이 왜 필요한가**
- **초기 단계 ≈ 선형 출생–사망 과정**: $e^{-(\beta - \gamma)t}I_t$가 마팅게일이고 $L^2$ 유계라 $W$로 수렴한다(05d의 $Z_n/m^n \to W$).
- **이후 단계는 거의 결정론적($N$ 큼)**: 그래서 '모양'은 ODE, '시작 시각'만 무작위다. 정점 근처의 추가 잡음과 정점 위치 탐색 잡음 때문에 MC sd(1.45)가 예측(1.28)보다 약간 크다(KS 통계 ≈ 0.05).

**증명 스케치.** 선형 출생–사망: $E[I_t] = e^{(\beta - \gamma)t}$, $M_t = e^{-(\beta - \gamma)t}I_t$는 마팅게일, $\sup_t E[M_t^2] < \infty$ → $M_t \to W$. $W$의 분포(생존 조건부 지수)는 pgf 미분방정식(Athreya–Ney Ch. III)에서 나온다. ODE에서 $I(0) = 1$이면 초기에 $I \approx e^{(\beta - \gamma)t}$; 확률 경로는 $We^{(\beta - \gamma)t} = e^{(\beta - \gamma)(t + \ln W/(\beta - \gamma))}$ → 시간 이동 $\delta = \ln W/(\beta - \gamma)$, 즉 $T = T^{ODE} - \delta$. $P(T \le t) = P(W \ge w) = e^{-w/\theta}$, $w = e^{-(\beta - \gamma)(t - T^{ODE})}$이므로 밀도는 $f_T(t) = (\beta - \gamma)\,w\,e^{-w/\theta}/\theta$ (Gumbel형). $E[\ln W] = \ln\theta - \gamma_E$, $\mathrm{Var}(\ln W) = \pi^2/6$.

### 2.6 정리 5 — 작은 $N$의 정확 계산 (Durrett 3e §4.2, §4.4; 03b/01c)

> $N = 30$: 상태 $(S, I)$ 496개, 생성원 $Q$. $P(t) = e^{Qt}$의 행 $(29, 1)$에서 $P(I_t = 0) = \sum_s P_t[(s, 0)]$: $t = 1$: 0.39710, 2: 0.49063, 5: 0.66260, 10: 0.97433; $E[I_5] = 1.0244$. 기대 소멸시간 $\tau$는 $-Q_{TT}\tau = \mathbf 1$의 해: 3.3408. $t \to \infty$의 흡수확률(각 $(s, 0)$)은 DP pmf와 일치한다.

**가정이 왜 필요한가**
- **유한 상태 공간**: 밀집 expm은 $N \lesssim 60$까지 실용적($N = 1000$은 50만 상태).
- **흡수 상태 $(s, 0)$가 여럿**: 최종 규모 = 어느 흡수 상태에 도달하는가(01c의 $B = NR$과 같은 정보).

증명은 인용: 03b의 전방 방정식 $P'(t) = P(t)Q$의 해 $e^{Qt}$; 01c의 기본행렬 $(-Q_{TT})^{-1}$의 행합이 기대 흡수시간. 수치는 단계 1에서 재현한다.

### 2.7 직관

전염병의 운명은 두 번 결정된다. 처음 몇 명이 감염될 때는 **소수의 법칙**(분기 과정)이 지배해 확률 $1/R_0$로 그냥 꺼진다; 감염자가 수십 명을 넘으면 **큰 수의 법칙**(ODE)이 지배해 최종 규모가 거의 확정된다(±29명). 그래서 최종 규모 분포는 이봉이고, ODE는 두 번째 봉우리만 본다. 정점 시각의 흩어짐은 첫 단계의 '운'($W$)이 두 번째 단계의 시계를 얼마나 앞당기거나 늦추는지의 문제다 — 05d의 $Z_n/m^n \to W$가 여기서 실제로 쓰인다. 작은 $N$에서는 이 모든 것이 496×496 행렬 하나로 정확히 나온다.

### 2.8 함정(traps)

- $R_0 = 2$라고 반드시 대유행이 나는 것이 아니다: $I_0 = 1$이면 절반(0.502)은 10명 남짓에서 끝난다. 반대로 대유행이 나면 규모는 거의 확정적(795 ± 29).
- $1/R_0$은 마르코프(지수) 감염기간의 값이다. 고정 감염기간이면 Poisson 자손 → 소멸확률 0.203. 최종 규모 방정식의 $z$는 감염기간 분포에 무관하다(평균만 중요) — 두 결과의 민감도가 다르다.
- ODE의 정점 시각(6.757)은 확률 정점 시각의 평균(6.63–6.64)이 아니고, 분산은 초기 단계에서 온다 — $N$을 키워도 정점 시각의 sd는 $\pi/(\sqrt6(\beta - \gamma)) = 1.28$로 남는다.
- 정확 pmf에서 $Z \in [51, 650]$의 확률이 $10^{-4}$ 미만이므로 minor/major 문턱(100)은 둔감하지만, $R_0$가 1에 가까우면($R_0 = 1.5$) 골이 얕아져 문턱에 민감해진다.
- MC 비율의 SE: $P(\text{minor})$ 추정에 $n = 20{,}000$이면 SE 0.0035 → $1/R_0$과 정확값의 차이 0.002를 구분하지 못한다(4 SE 판정은 둘 다 통과). 차이를 보려면 DP(정확)를 써라.
- 정점 $I_{max}$의 MC 평균(≈167)이 ODE(153.9)보다 큰 것은 잡음 경로의 최댓값이 위로 치우치기 때문 — 버그가 아니다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요)

## 3. Predict — 코드를 돌리기 전에 예측

숫자를 적어 보세요. Compare 표의 '내 예측' 열에 들어갑니다(채점 대상은 아닙니다).

In [ ]:
predictions = {
    "p_minor": None,               # N=1000, I0=1, R0=2: 소규모 유행(Z<=100)으로 끝날 확률(정확값)은? 분기 근사 1/R0 = 0.5보다 클까 작을까
    "p_minor_I0_3": None,          # 초기 감염자가 3명이면 소규모 유행 확률은?
    "final_size_major": None,      # 대유행이 났을 때 총 감염자 수 E[Z | Z>100]은? (ODE 최종 규모와 비교)
    "final_size_sd_major": None,   # 그때의 표준편차는? (몇 명 정도 흔들릴까)
    "peak_time_ode": None,         # ODE(S0=999, I0=1)의 정점 시각은? (시간 단위 = 평균 감염기간)
    "peak_time_mc_mean": None,     # 확률 모형(대유행 조건부)의 정점 시각 평균은? ODE보다 앞일까 뒤일까
    "p_first_event_infection": None,  # 상태 (999, 1)에서 다음 사건이 감염일 확률은?
    "N30_p_I5_zero": None,         # N=30, (29,1)에서 시작: P(I_5 = 0)은?
    "N30_E_T_ext": None,           # N=30: 기대 소멸시간 E[T_ext]은?
    "q10_gw": None,                # 기하(p=1/3) 자손 GW 과정이 10세대 안에 소멸할 확률은? (무한 세대: 1/2)
    "p_minor_R0_1_5": None,        # R0=1.5이면 소규모 유행 확률은? (1/R0 = 0.667과 비교)
}

## 4. 단계 1 — 작은 $N$ 정확 계산: 생성원, expm, 흡수확률, Gillespie (03a·03b·01c 재사용)

$N = 30$이면 상태 $(S, I)$, $S + I \le N$는 $\binom{32}{2} = 496$개 — 밀집 생성원 하나로 03b의 $e^{Qt}$, 01c의 흡수확률·기대 흡수시간을 모두 정확히 계산할 수 있다. 상태 순서는 $(S, I)$ 사전순.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def sir_generator(N, beta, gamma):
    """마르코프 SIR의 밀집 생성원. 반환: (Q, index[(S,I)] -> k, states)."""
    states = [(S, I) for S in range(N + 1) for I in range(N + 1 - S)]
    index = {st: k for k, st in enumerate(states)}
    Q = np.zeros((len(states), len(states)))
    for (S, I), k in index.items():
        if I == 0:                       # 흡수 상태
            continue
        a_inf, a_rec = beta * S * I / N, gamma * I
        if S > 0:
            Q[k, index[(S - 1, I + 1)]] = a_inf
        Q[k, index[(S, I - 1)]] = a_rec
        Q[k, k] = -(a_inf * (S > 0) + a_rec)
    return Q, index, states

N30 = 30
Q30, idx30, states30 = sir_generator(N30, BETA, GAMMA)
I_of = np.array([I for S, I in states30])                 # 상태 k의 I 값
x0_30 = idx30[(N30 - 1, 1)]
absorbing30 = np.array([idx30[(s, 0)] for s in range(N30 + 1)])
assert is_generator(Q30) and Q30.shape == (496, 496)

J30, q30 = embedded_chain(Q30)                            # 점프 사슬 (정리 1)
p_jump = J30[x0_30, idx30[(N30 - 2, 2)]]
print(f"states={len(states30)}, jump-chain P(infection | (29,1)) = {p_jump:.6f} = R0*29/(R0*29+30) = {R0 * 29 / (R0 * 29 + 30):.6f}")

T_LIST = [1, 2, 5, 10, 20]
exact30 = {}
for t in T_LIST:
    row = transition_matrix(Q30, float(t))[x0_30]
    exact30[t] = (row[absorbing30].sum(), row @ I_of)
    print(f"t={t:>2}: P(I_t = 0) = {exact30[t][0]:.5f},  E[I_t] = {exact30[t][1]:.4f}")
P_I5_ZERO, E_I5 = exact30[5]

`transition_matrix(Q, t)`(= $e^{Qt}$, 03b)의 행 $(29, 1)$을 흡수 상태 $(s, 0)$ 위에서 합하면 $P(I_t = 0)$, $I$ 값과 내적하면 $E[I_t]$다. 이제 정리 1의 DP를 구현하고, $t = 300$의 흡수확률과 $10^{-10}$ 이내로 맞는지, 기대 소멸시간은 01c의 기본행렬로 얼마인지 본다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def sir_final_size_pmf(N, I0, R0, S0=None):
    """점프 사슬 DP (정리 1). 반환: pmf over S_final = s (index s = 0..S0); 최종 규모 Z = S0 + I0 - s."""
    S0 = N - I0 if S0 is None else S0
    f = np.zeros((S0 + 1, S0 + I0 + 2))
    f[S0, I0] = 1.0
    for s in range(S0, -1, -1):
        p = R0 * s / (R0 * s + N)
        for i in range(S0 + I0 - s, 0, -1):      # I 내림차순: (s, i+1)의 질량이 먼저 도착
            m = f[s, i]
            if m == 0.0:
                continue
            if s > 0:
                f[s - 1, i + 1] += m * p
            f[s, i - 1] += m * (1 - p)
    return f[:, 0]

pmf30 = sir_final_size_pmf(N30, 1, R0)                      # index s = 0..29
Z30_vals = N30 - np.arange(N30)                             # Z = N - s (I0 = 1)
absorb_expm = transition_matrix(Q30, 300.0)[x0_30, absorbing30]   # (s,0), s = 0..30; (30,0)은 도달 불가
np.testing.assert_allclose(absorb_expm[:N30], pmf30, atol=1e-10)
P_Z30_LE5 = pmf30[Z30_vals <= 5].sum()
print(f"expm absorption vs DP: max|diff| = {np.abs(absorb_expm[:N30] - pmf30).max():.1e},  P(Z <= 5) = {P_Z30_LE5:.5f}")

transient30 = np.array([k for k, (S, I) in enumerate(states30) if I > 0])
tau = np.linalg.solve(-Q30[np.ix_(transient30, transient30)], np.ones(transient30.size))   # 01c: (-Q_TT)^{-1} 1
E_T_EXT = tau[np.searchsorted(transient30, x0_30)]
np.testing.assert_allclose(P_I5_ZERO, 0.66260, atol=1e-5)
np.testing.assert_allclose(E_T_EXT, 3.3408, atol=1e-4)
print(f"E[T_ext | (29,1)] = {E_T_EXT:.4f}")

Gillespie로 같은 것을 표본화한다. 그림용 20개 경로는 03a의 `gillespie`(생성원 $Q$ 하나로 한 경로씩)로 뽑는다. 그러나 $P(I_5 = 0)$ 같은 비율을 SE 0.001 수준으로 재려면 경로 수십만 개가 필요한데, `gillespie`는 호출마다 496상태 점프 사슬을 다시 만들어 한 경로에 약 5 ms가 걸린다. 그래서 SIR 전용 **벡터화 Gillespie**를 정의한다: 배열 $(S, I, t)$를 경로 축으로 한꺼번에 갱신하고(체류시간 Exp($a$), 사건 유형 Bernoulli($a_{inf}/a$) — 02a), $I = 0$인 경로는 비활성화한다. 시각 $t_{obs}$를 덮는 체류 구간($t_{old} \le t_{obs} < t_{new}$)의 상태가 $I_{t_{obs}}$이고(그 전에 흡수된 경로는 0), 마지막 점프 시각이 소멸시각이다. 이 helper는 단계 2의 $N = 1000$에서도 그대로 쓴다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def sir_gillespie_batch(N, beta, gamma, I0, n_paths, rng, t_obs=None):
    """경로 축 벡터화 Gillespie. 반환: (final_size Z, peak_I, t_peak, t_end, I_obs = I_{t_obs})."""
    S = np.full(n_paths, N - I0, dtype=np.int64)
    I = np.full(n_paths, I0, dtype=np.int64)
    t = np.zeros(n_paths)
    peak_I, t_peak = I.copy(), np.zeros(n_paths)
    I_obs = np.zeros(n_paths, dtype=np.int64)              # t_obs 전에 흡수된 경로는 0으로 남는다
    active = np.arange(n_paths)
    while active.size:
        a_inf = beta * S[active] * I[active] / N
        a = a_inf + gamma * I[active]
        t_new = t[active] + rng.exponential(1.0 / a)           # 체류시간
        if t_obs is not None:                                  # 체류 구간이 t_obs를 덮으면 지금 상태가 I_{t_obs}
            hit = active[(t[active] <= t_obs) & (t_obs < t_new)]
            I_obs[hit] = I[hit]
        t[active] = t_new
        inf = rng.random(active.size) < a_inf / a              # 감염 vs 회복 (02a)
        S[active] -= inf
        I[active] += np.where(inf, 1, -1)
        up = active[I[active] > peak_I[active]]                # 새 최댓값 (첫 도달 시각)
        peak_I[up], t_peak[up] = I[up], t[up]
        active = active[I[active] > 0]                         # I = 0 이면 흡수
    return N - S, peak_I, t_peak, t, I_obs

In [ ]:
t0 = time.perf_counter()
Z30_mc, _, _, T_ext, I5 = sir_gillespie_batch(N30, BETA, GAMMA, 1, N_SMALL, rng, t_obs=5.0)
t_small = time.perf_counter() - t0
mc_I5_zero, mc_I5, mc_T_ext = mc_proportion(I5 == 0), mc_estimate(I5), mc_estimate(T_ext)
mc_Z30_le5 = mc_proportion(Z30_mc <= 5)
print(f"vectorized Gillespie, N = 30, {N_SMALL} paths: {t_small:.2f} s")
print(f"P(I_5=0): {mc_I5_zero}  exact {P_I5_ZERO:.5f}\nE[I_5]:   {mc_I5}  exact {E_I5:.4f}")
print(f"E[T_ext]: {mc_T_ext}  exact {E_T_EXT:.4f}\nP(Z<=5):  {mc_Z30_le5}  exact {P_Z30_LE5:.5f}")

paths30 = [gillespie(Q30, x0_30, 15.0, rng) for _ in range(20)]     # 03a: 그림용 20개 경로
t_grid = np.linspace(0, 15, 31)
E_I_curve = [transition_matrix(Q30, t)[x0_30] @ I_of for t in t_grid]
fig, ax = plt.subplots()
for times, sts in paths30:
    I_path = I_of[sts]
    ax.step(np.append(times, 15.0), np.append(I_path, I_path[-1]), where="post", alpha=0.5, lw=0.9, color="C0")
ax.plot(t_grid, E_I_curve, color="C1", lw=2.5, label="exact E[I_t] (expm)")
ax.set(xlim=(0, 15), xlabel="t", ylabel="I(t)", title="N = 30 stochastic SIR: 20 Gillespie paths of I(t)")
ax.legend();

네 통계량이 모두 정확값의 2 SE 안에 들어오는지 보라(SE는 각각 0.001, 0.004, 0.007 수준). 경로 수를 1,000으로 줄이면 SE가 4–5배 커져 같은 시드에서도 2–3 SE 요동이 흔히 보인다 — 그것은 버그가 아니라 표본 오차이고, 4 SE 판정은 그래서 있다.

## 5. 단계 2 — $N = 1000$ Gillespie와 최종 규모의 이봉 분포

$N = 1000$이면 상태가 50만 개라 expm은 못 쓰지만 (a) DP는 여전히 $O(N^2)$이고 (b) 단계 1의 벡터화 Gillespie는 경로당 사건 수가 늘어도 반복 횟수(최대 사건 수 ≈ $2N$)만 늘어 2만 경로에 1초 남짓이다. 배치는 정점 $I_{max}$와 그 시각, 종료 시각도 함께 돌려준다. 그림용으로 모든 점프를 저장하는 단일 경로 helper만 따로 둔다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def sir_gillespie_path(N, beta, gamma, I0, rng):
    """그림용 단일 경로. 반환: (times, S, I) — 모든 점프."""
    S, I, t = N - I0, I0, 0.0
    times, Ss, Is = [0.0], [S], [I]
    while I > 0:
        a_inf = beta * S * I / N
        a = a_inf + gamma * I
        t += rng.exponential(1.0 / a)
        if rng.random() < a_inf / a:
            S, I = S - 1, I + 1
        else:
            I -= 1
        times.append(t); Ss.append(S); Is.append(I)
    return np.array(times), np.array(Ss), np.array(Is)

In [ ]:
t0 = time.perf_counter()
Z, peak_I, t_peak, t_end, _ = sir_gillespie_batch(N, BETA, GAMMA, I0, N_PATHS, rng)
t_batch = time.perf_counter() - t0
major = Z > MINOR
mc_p_minor, mc_Z_major = mc_proportion(Z <= MINOR), mc_estimate(Z[major])

t0 = time.perf_counter()
pmf1000 = sir_final_size_pmf(N, I0, R0)                    # index s = 0..999
t_dp = time.perf_counter() - t0
Z_vals = N - np.arange(N - I0 + 1)                          # Z = N - s
np.testing.assert_allclose(pmf1000.sum(), 1.0, atol=1e-10)
P_MINOR_EXACT = pmf1000[Z_vals <= MINOR].sum()
w = pmf1000[Z_vals > MINOR] / pmf1000[Z_vals > MINOR].sum()
E_Z_MAJOR = w @ Z_vals[Z_vals > MINOR]
SD_Z_MAJOR = np.sqrt(w @ Z_vals[Z_vals > MINOR] ** 2 - E_Z_MAJOR**2)
Z_MODE = Z_vals[Z_vals > MINOR][np.argmax(w)]
TROUGH = pmf1000[(Z_vals > 50) & (Z_vals <= 650)].sum()
assert TROUGH < 1e-3 and abs(P_MINOR_EXACT - 1 / R0) < 0.01
print(f"Gillespie {N_PATHS} paths: {t_batch:.2f} s;  DP N=1000: {t_dp:.2f} s")
print(f"P(minor) MC {mc_p_minor}  exact {P_MINOR_EXACT:.6f}  (1/R0 = {1 / R0})")
print(f"E[Z|major] MC {mc_Z_major}  exact {E_Z_MAJOR:.3f} (sd {SD_Z_MAJOR:.3f}, mode {Z_MODE}),  E[Z] = {pmf1000 @ Z_vals:.2f}")
print(f"trough mass P(50 < Z <= 650) = {TROUGH:.1e}")

두 번째 봉우리(정확 pmf: 평균 795.4, sd 29.4)를 결정론적 예측과 견주기 위해 먼저 SIR ODE($S_0 = 999$, $I_0 = 1$)를 풀고, 최종 규모 방정식 $z = 1 - e^{-R_0 z}$의 해를 `brentq`로 구한다(정리 3).

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def sir_ode(N, beta, gamma, I0, T=40.0):
    rhs = lambda t, y: [-beta * y[0] * y[1] / N, beta * y[0] * y[1] / N - gamma * y[1]]
    return solve_ivp(rhs, (0.0, T), [N - I0, I0], dense_output=True, rtol=1e-10, atol=1e-10)

def ode_peak(sol, T=20.0, dt=1e-4):
    tg = np.arange(0.0, T, dt)
    Ig = sol.sol(tg)[1]
    k = np.argmax(Ig)
    return tg[k], Ig[k]

def final_size_z(R0):
    return brentq(lambda z: z - 1 + np.exp(-R0 * z), 1e-9, 1.0)

sol = sir_ode(N, BETA, GAMMA, I0)
T_PEAK_ODE, I_PEAK_ODE = ode_peak(sol)
ODE_FINAL = N - sol.sol(40.0)[0]
Z_EQ = final_size_z(R0)
I_PEAK_ASYMP = N * (1 - 1 / R0 - np.log(R0) / R0)
np.testing.assert_allclose(Z_EQ, 0.796812, atol=1e-6)
assert abs(ODE_FINAL / N - 0.797154) < 1e-4 and abs(T_PEAK_ODE - 6.7573) < 5e-3
print(f"ODE: peak time {T_PEAK_ODE:.4f}, peak I {I_PEAK_ODE:.2f} (asymptotic {I_PEAK_ASYMP:.2f}), final size {ODE_FINAL:.2f} = {ODE_FINAL / N:.6f} N;  z(R0) = {Z_EQ:.6f}, N z = {N * Z_EQ:.1f}")

이제 최종 규모 히스토그램에 정확 pmf(50명 구간으로 합침)를 겹치고 ODE 최종 규모를 점선으로 넣는다. 로그 축이라 두 봉우리 사이의 빈 골이 보인다.

In [ ]:
edges = np.arange(0, N + 51, 50)
pmf_binned = np.array([pmf1000[(Z_vals >= a) & (Z_vals < b)].sum() for a, b in zip(edges[:-1], edges[1:])])
fig, ax = plt.subplots()
ax.hist(Z, bins=edges, density=True, alpha=0.5, label=f"Gillespie ({N_PATHS} paths)")
ax.step(edges, np.append(pmf_binned, pmf_binned[-1]) / 50, where="post", color="C1", lw=2, label="exact pmf (DP), binned")
ax.axvline(ODE_FINAL, color="k", ls="--", lw=1, label=f"ODE final size {ODE_FINAL:.0f}")
ax.set(xlabel="final size Z", ylabel="density", yscale="log", ylim=(1e-6, 0.05),
       title=f"Final size of stochastic SIR (N = {N}, R0 = {R0:g}, I0 = {I0}): bimodal")
ax.legend();

두 봉우리가 $Z \le 50$과 800 근처에 있고 그 사이의 골은 $10^{-4}$ 미만이다; ODE 최종 규모(797)는 두 번째 봉우리 위에 놓이고 첫 봉우리는 전혀 보지 못한다. 이제 유행 곡선 자체를 결정론적 ODE와 비교한다 — 30개 경로의 $I(t)$ 위에 ODE($S_0 = 999$, $I_0 = 1$)를 굵게 그린다.

In [ ]:
fig, ax = plt.subplots()
for j in range(30):
    times, Ss, Is = sir_gillespie_path(N, BETA, GAMMA, I0, rng)
    ax.step(times, Is, where="post", alpha=0.45, lw=0.8, color="C0")
tg = np.linspace(0, 25, 500)
ax.plot(tg, sol.sol(tg)[1], color="C1", lw=3, label="deterministic ODE (S0 = 999, I0 = 1)")
ax.axvline(T_PEAK_ODE, color="C1", ls="--", lw=1)
ax.set(xlabel="t", ylabel="I(t)", xlim=(0, 25), title="30 Gillespie sample paths of I(t) vs ODE (N = 1000, R0 = 2)")
ax.legend();

경로들은 ODE와 '모양'이 같지만 시간축에서 좌우로 밀려 있고(정리 4), 절반가량은 바닥에서 바로 꺼진다. 마지막으로 정리 1의 출발점인 02a의 경쟁하는 시계를 직접 확인한다: 상태 $(999, 1)$에서 감염 시계 Exp($\beta \cdot 999/N$)와 회복 시계 Exp($\gamma$) 중 감염이 먼저일 확률.

In [ ]:
n_pairs = scale(20_000)
a_inf0 = BETA * (N - I0) / N
first_inf = rng.exponential(1 / a_inf0, n_pairs) < rng.exponential(1 / GAMMA, n_pairs)
mc_first_inf = mc_proportion(first_inf)
P_FIRST_INF = a_inf0 / (a_inf0 + GAMMA)
print(f"P(first event is infection | (999,1)): MC {mc_first_inf}, exact {P_FIRST_INF:.6f}, branching limit R0/(1+R0) = {R0 / (1 + R0):.6f}")

## 6. 단계 3 — 분기 과정 근사 (01g·02a 재사용)

초기 단계의 감염자 한 명은 기하 자손 $K \sim \mathrm{Geom}(p)$, $p = 1/(1 + R_0) = 1/3$을 남긴다(정리 2). 01g의 `extinction_probability`(pgf 고정점 반복)와 `galton_watson`(세대별 시뮬레이션)으로 $q_\infty = 1/R_0$, $q_{10}$을 확인하고, $I_0 = 3$의 $q^3$을 정확 SIR DP·Gillespie와 대조한다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def geometric_offspring_pmf(R0, k_max=60):
    p = 1 / (1 + R0)
    return (1 - p) ** np.arange(k_max + 1) * p

def gw_extinct_by_generation(pmf, n):
    """q_n = f^{on}(0), f(s) = sum_k pmf[k] s^k."""
    s = 0.0
    for _ in range(n):
        s = np.polynomial.polynomial.polyval(s, pmf)
    return s

geom_pmf = geometric_offspring_pmf(R0)
assert 1 - geom_pmf.sum() < 1e-10
Q_INF = extinction_probability(geom_pmf)
np.testing.assert_allclose(Q_INF, 1 / R0, atol=1e-9)
Q10 = gw_extinct_by_generation(geom_pmf, 10)
print("q_n for n = 1, 2, 3, 5, 10, 15:", ", ".join(f"{gw_extinct_by_generation(geom_pmf, n):.5f}" for n in (1, 2, 3, 5, 10, 15)), f"; q_inf = {Q_INF:.9f}")

gw = galton_watson(geom_pmf, 10, rng, n_paths=N_PATHS)
mc_q10 = mc_proportion(gw[:, -1] == 0)
gw3 = galton_watson(geom_pmf, 10, rng, n_paths=N_PATHS, z0=3)
mc_q10_3 = mc_proportion(gw3[:, -1] == 0)

pmf1000_3 = sir_final_size_pmf(N, 3, R0)                   # I0 = 3: Z = N - s, s = 0..997
P_MINOR_EXACT_3 = pmf1000_3[(N - np.arange(N - 3 + 1)) <= MINOR].sum()
Z_I03 = sir_gillespie_batch(N, BETA, GAMMA, 3, N_PATHS, rng)[0]
mc_p_minor_3 = mc_proportion(Z_I03 <= MINOR)
print(f"GW extinct by gen 10: MC {mc_q10}  exact {Q10:.5f};   z0 = 3: MC {mc_q10_3}  exact q10^3 = {Q10**3:.5f}")
print(f"SIR I0 = 3, P(minor): MC {mc_p_minor_3}  exact {P_MINOR_EXACT_3:.6f}  branching (1/R0)^3 = {(1 / R0)**3}")

소규모 유행의 크기 분포까지 분기 과정이 맞추는지 본다. 총 자손(total progeny) $Y$ = 계통의 총 개체 수(조상 포함)는 기하 자손에서 닫힌 형식 $P(Y = k) = \binom{2k - 2}{k - 1}\frac1k\,p^k(1 - p)^{k - 1}$(Dwass 공식)이고, 정확 SIR pmf의 $Z = k$($k \le 20$)와 비교한다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def bp_total_progeny_pmf(p, k_max):
    k = np.arange(1, k_max + 1)
    return k, comb(2 * k - 2, k - 1) / k * p**k * (1 - p) ** (k - 1)

k_small, bp_pmf = bp_total_progeny_pmf(1 / (1 + R0), 20)
sir_small = np.array([pmf1000[Z_vals == k][0] for k in k_small])
hdr = "| $k$ | SIR exact $P(Z = k)$ | branching $P(Y = k)$ | ratio |\n|---|---|---|---|\n"
tbl = "\n".join(f"| {k} | {sir_small[k - 1]:.5f} | {bp_pmf[k - 1]:.5f} | {sir_small[k - 1] / bp_pmf[k - 1]:.4f} |" for k in (1, 2, 3, 5, 10, 20))
fig, ax = plt.subplots()
ax.bar(k_small - 0.2, sir_small, width=0.4, label="exact SIR pmf (N = 1000)")
ax.bar(k_small + 0.2, bp_pmf, width=0.4, label="branching total progeny (p = 1/3)")
ax.set(xlabel="outbreak size k", ylabel="probability", yscale="log", xticks=k_small,
       title="Minor-outbreak sizes: exact SIR vs branching process")
ax.legend()
Markdown(hdr + tbl + f"\n\n$\\sum_{{k \\le 20}}$: SIR {sir_small.sum():.5f}, branching {bp_pmf.sum():.5f}; branching total minor mass $= 1/R_0 = {1 / R0}$ vs SIR {P_MINOR_EXACT:.6f}.")

## 7. 단계 4 — 정점 시각 분포와 시간 이동 법칙 (05d 재사용)

단계 2의 배치에서 대유행 경로의 정점 시각 $T_{peak}$를 모아 히스토그램을 그리고, 정리 4의 밀도 $f_T(t) = (\beta - \gamma)\,w\,e^{-w/\theta}/\theta$, $w = e^{-(\beta - \gamma)(t - T^{ODE})}$, $\theta = \beta/(\beta - \gamma)$를 겹친다. 이 단계는 참고값 비교만 한다(assert 없음 — 법칙은 근사다).

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def peak_time_shift_density(t, t_ode, beta, gamma):
    r, theta = beta - gamma, beta / (beta - gamma)
    w = np.exp(-r * (t - t_ode))
    return r * w * np.exp(-w / theta) / theta

r, theta = BETA - GAMMA, BETA / (BETA - GAMMA)
law = {"mean": T_PEAK_ODE - (np.log(theta) - np.euler_gamma) / r, "sd": np.pi / (np.sqrt(6) * r),
       "median": T_PEAK_ODE - np.log(theta * np.log(2)) / r,
       "q10": T_PEAK_ODE - np.log(theta * np.log(10)) / r, "q90": T_PEAK_ODE - np.log(theta * np.log(10 / 9)) / r}
tp = t_peak[major]
mc_t_peak = mc_estimate(tp)
mc = {"mean": tp.mean(), "sd": tp.std(ddof=1), "median": np.median(tp), "q10": np.quantile(tp, 0.1), "q90": np.quantile(tp, 0.9)}
ks = ks_1samp(tp, lambda t: np.exp(-np.exp(-r * (t - T_PEAK_ODE)) / theta)).statistic
mc_peak_I = mc_estimate(peak_I[major])
dur_major, dur_minor = t_end[major].mean(), t_end[~major].mean()
print(f"time to peak | major: MC {mc_t_peak} (sd {mc['sd']:.3f})  vs law mean {law['mean']:.3f}, sd {law['sd']:.3f};  KS = {ks:.3f}")

밀도와 히스토그램을 겹쳐 그리고(안쪽 그림은 정점 크기 vs ODE 정점), 평균·sd·중앙값·10/90% 분위수를 표로 나란히 놓는다.

In [ ]:
tg = np.linspace(2, 14, 1000)
fig, ax = plt.subplots()
plot_hist_vs_pdf(tp, tg, peak_time_shift_density(tg, T_PEAK_ODE, BETA, GAMMA), ax=ax, bins=40, label_pdf="time-shift law density")
ax.axvline(T_PEAK_ODE, color="k", ls="--", lw=1, label=f"ODE peak time {T_PEAK_ODE:.3f}")
ax.set(xlabel="time to peak", ylabel="density", title=f"Time to peak given a major outbreak (N = {N}, R0 = {R0:g})")
ax.legend(loc="upper right")
ins = ax.inset_axes([0.62, 0.35, 0.35, 0.3])
ins.hist(peak_I[major], bins=30, density=True, alpha=0.6)
ins.axvline(I_PEAK_ODE, color="C1", lw=1.5)
ins.set(title="peak size vs ODE", xlabel="max I")
Markdown("| 통계량 | MC (major) | 시간 이동 법칙 |\n|---|---|---|\n" +
         "\n".join(f"| {k} | {mc[k]:.3f} | {law[k]:.3f} |" for k in ("mean", "sd", "median", "q10", "q90")) +
         f"\n\nKS 통계 (MC vs 법칙) = {ks:.3f}; 정점 크기 MC {mc_peak_I.mean:.1f} (sd {peak_I[major].std(ddof=1):.1f}) vs ODE {I_PEAK_ODE:.1f}; "
         f"지속 시간 평균: major {dur_major:.1f}, minor {dur_minor:.2f}.")

법칙은 위치(평균·중앙값)를 FULL 모드(2만 경로)에서 0.02 안에서 맞추지만(FAST 모드는 대유행 경로가 1,000개 남짓이라 평균의 SE가 0.05다) MC sd(≈1.45)가 예측(1.28)보다 크다 — 정점 근처의 잡음(경로의 최댓값이 어디서 찍히는지)이 초기 단계의 $W$ 외에 분산을 더한다. 정점 크기 평균이 ODE보다 큰 것(≈167 vs 154)도 같은 '최댓값의 위쪽 치우침'이다.

## 8. 단계 5 — $R_0$ 민감도: 소멸확률과 최종 규모

DP는 $R_0$마다 0.5초이므로 $R_0 \in \{1.2, 1.5, 2, 2.5, 3, 4\}$의 정확 $P(\text{minor})$와 $E[Z \mid \text{major}]/N$을 계산하고, 분기 근사 $1/R_0$·최종 규모 방정식 $z(R_0)$과 나란히 그린다. MC는 $R_0 = 1.5$(FULL 모드에서는 3.0도).

In [ ]:
R0_DP = [1.2, 1.5, 2.0, 2.5, 3.0, 4.0]
dp_rows = {}
for r0 in R0_DP:
    pm = sir_final_size_pmf(N, I0, r0)
    mj = Z_vals > MINOR
    ww = pm[mj] / pm[mj].sum()
    e = ww @ Z_vals[mj]
    dp_rows[r0] = (pm[~mj].sum(), e / N, np.sqrt(ww @ Z_vals[mj] ** 2 - e**2), pm[(Z_vals > 50) & (Z_vals <= 0.6 * e)].sum())
mc_R0 = {}
for r0 in R0_MC:
    Zr = sir_gillespie_batch(N, r0 * GAMMA, GAMMA, I0, N_PATHS, rng)[0]
    mc_R0[r0] = (mc_proportion(Zr <= MINOR), mc_estimate(Zr[Zr > MINOR] / N), Zr)
print("R0    P(minor)  1/R0    E[Z|maj]/N  z(R0)    sd(Z|maj)  trough(50, 0.6 E[Z|maj]]")
for r0, (pmn, ez, sd, tr) in dp_rows.items():
    print(f"{r0:<5} {pmn:.6f}  {1 / r0:.4f}  {ez:.5f}     {final_size_z(r0):.6f} {sd:7.1f}     {tr:.4f}")
for r0, (pmn, ez, _) in mc_R0.items():
    print(f"MC R0={r0}: P(minor) {pmn}, E[Z|major]/N {ez}")

In [ ]:
r0_fine = np.linspace(0.8, 4.0, 200)
z_fine = np.array([final_size_z(r0) if r0 > 1 else 0.0 for r0 in r0_fine])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(r0_fine, np.minimum(1, 1 / r0_fine), color="C1", label="branching approx. 1/R0")
ax1.plot(R0_DP, [v[0] for v in dp_rows.values()], "o", color="C0", label="exact CTMC (DP), N = 1000")
ax1.errorbar(list(mc_R0), [v[0].mean for v in mc_R0.values()], yerr=[1.96 * v[0].se for v in mc_R0.values()], fmt="s", color="C2", capsize=3, label="MC (95% CI)")
ax1.set(xlabel="R0", ylabel="P(minor outbreak)", title="Probability of early extinction")
ax1.legend()
ax2.plot(r0_fine, z_fine, color="C1", label="final-size equation z(R0)")
ax2.plot(R0_DP, [v[1] for v in dp_rows.values()], "o", color="C0", label="exact E[Z | major] / N (DP)")
ax2.errorbar(list(mc_R0), [v[1].mean for v in mc_R0.values()], yerr=[1.96 * v[1].se for v in mc_R0.values()], fmt="s", color="C2", capsize=3, label="MC (95% CI)")
ax2.set(xlabel="R0", ylabel="conditional final-size fraction", title="Size of a major outbreak")
ax2.legend()
fig.tight_layout()

Z15 = mc_R0[1.5][2]
fig, ax = plt.subplots()
ax.hist(Z15, bins=np.arange(0, N + 26, 25), density=True, alpha=0.6, label="Gillespie, R0 = 1.5")
ax.hist(Z, bins=np.arange(0, N + 26, 25), density=True, alpha=0.4, label="Gillespie, R0 = 2")
ax.set(xlabel="final size Z", ylabel="density", yscale="log", ylim=(1e-5, 0.1), title="Final-size histograms: shallow trough at R0 = 1.5")
ax.legend();

$R_0 = 1.5$에서는 대유행 봉우리가 578 ± 66으로 넓고 낮아져 골이 얕다(50 < Z ≤ 350 사이 질량 ≈ 0.5%, $R_0 = 2$에서는 $10^{-4}$ 미만). $R_0 = 4$에서는 $P(\text{minor})$가 $1/R_0$과 $5 \times 10^{-4}$ 차이로 붙고 최종 규모는 $0.98N$이다.

## 9. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

In [ ]:
P = predictions
rows = [
    {"name": "N=30: P(I_5 = 0) (Gillespie vs expm)", "predicted": P["N30_p_I5_zero"], "estimate": mc_I5_zero, "exact": P_I5_ZERO},
    {"name": "N=30: E[I_5]", "predicted": None, "estimate": mc_I5, "exact": E_I5},
    {"name": "N=30: E[extinction time] from (29,1)", "predicted": P["N30_E_T_ext"], "estimate": mc_T_ext, "exact": E_T_EXT},
    {"name": "N=30: P(Z <= 5) (vectorized Gillespie vs absorption pmf)", "predicted": None, "estimate": mc_Z30_le5, "exact": P_Z30_LE5},
    {"name": "N=1000, R0=2, I0=1: P(Z <= 100) (minor outbreak) [1/R0 = 0.5]", "predicted": P["p_minor"], "estimate": mc_p_minor, "exact": P_MINOR_EXACT},
    {"name": f"N=1000, R0=2: E[Z | Z > 100] [ODE {ODE_FINAL:.1f}, N z = {N * Z_EQ:.1f}]", "predicted": P["final_size_major"], "estimate": mc_Z_major, "exact": E_Z_MAJOR},
    {"name": "N=1000, R0=2, I0=3: P(Z <= 100) [(1/R0)^3 = 0.125]", "predicted": P["p_minor_I0_3"], "estimate": mc_p_minor_3, "exact": P_MINOR_EXACT_3},
    {"name": "GW geometric(p=1/3): P(extinct by generation 10)", "predicted": P["q10_gw"], "estimate": mc_q10, "exact": Q10},
    {"name": "GW geometric, z0 = 3: P(extinct by generation 10)", "predicted": None, "estimate": mc_q10_3, "exact": Q10**3},
    {"name": "P(first event is an infection) at (999, 1)", "predicted": P["p_first_event_infection"], "estimate": mc_first_inf, "exact": P_FIRST_INF},
]
for r0, (pmn, ez, _) in mc_R0.items():
    rows.append({"name": f"N=1000, R0={r0}: P(Z <= 100) [1/R0 = {1 / r0:.4f}]", "predicted": P["p_minor_R0_1_5"] if r0 == 1.5 else None, "estimate": pmn, "exact": dp_rows[r0][0]})
    rows.append({"name": f"N=1000, R0={r0}: E[Z | major]/N [z = {final_size_z(r0):.4f}]", "predicted": None, "estimate": ez, "exact": dp_rows[r0][1]})
rows.append({"name": f"N=1000, R0=2: mean time to peak | major [law {law['mean']:.3f}, ODE {T_PEAK_ODE:.3f}] (reference only)", "predicted": P["peak_time_mc_mean"], "estimate": mc_t_peak, "exact": None})
Markdown(compare_table(rows))

행별로 '불일치'가 뜻하는 것:
- **N=30 네 행**: `sir_generator`의 율(감염 $\beta SI/N$ vs 회복 $\gamma I$, 대각 성분)이나 상태 인덱싱이 틀렸거나, `sir_gillespie_batch`가 $I_{t_{obs}}$를 읽는 조건($t_{old} \le t_{obs} < t_{new}$; 그 전에 흡수된 경로는 0)이 잘못된 것. $E[T_{ext}]$만 어긋나면 소멸시각을 '$I = 0$으로 들어가는 점프'가 아닌 다른 점프로 읽은 것. 네 통계량은 같은 경로에서 나온 상관된 값이라 한 번의 요동으로 함께 움직인다 — 그래서 경로 수를 20만(FAST 2만)으로 두어 SE를 0.001 수준으로 낮췄다.
- **P(minor), I0=3**: 벡터화 Gillespie의 사건 유형 확률 $a_{inf}/a$나 비활성화 조건. DP 쪽이 틀렸다면 정리 1의 전파 순서($S$·$I$ 내림차순).
- **E[Z|major]**: 최종 규모를 $N - S_\infty$가 아니라 $N - S_\infty - I_0$로 셌거나 문턱을 다르게 적용한 것. 조건부 표본 수가 $\approx N_{PATHS}/2$인지 확인.
- **GW 두 행**: `geometric_offspring_pmf`의 $p = 1/(1 + R_0)$ 또는 `gw_extinct_by_generation`의 반복 횟수(10세대 = 10번 합성).
- **첫 사건 감염**: 두 지수 시계의 평균을 율의 역수로 넣었는지($\beta \cdot 999/N$, $\gamma$).
- **R0=1.5 (및 3.0)**: DP와 배치 모두 $\beta = R_0\gamma$로 넣었는지. $R_0 = 1.5$는 골이 얕아 문턱 100에 민감하므로 DP와 MC가 같은 문턱을 써야 한다.
- **정점 시각**: 판정 '—'. FULL 모드에서는 법칙(6.64) 근처(≈ 6.63)여야 하지만 FAST 모드는 SE ≈ 0.05라 평균만으로는 판정이 안 된다 — sd(≈ 1.45, 법칙 1.28)와 히스토그램의 오른쪽 꼬리(Gumbel형)를 함께 보라. 시간 이동이 빠졌다면(초기 단계를 잘못 다룬 것) sd가 1보다 훨씬 작고 히스토그램이 ODE 정점 6.76에 몰린다.

In [ ]:
assert_close(mc_I5_zero, P_I5_ZERO, k=4, name="N=30 P(I_5=0)")
assert_close(mc_I5, E_I5, k=4, name="N=30 E[I_5]")
assert_close(mc_T_ext, E_T_EXT, k=4, name="N=30 E[T_ext]")
assert_close(mc_Z30_le5, P_Z30_LE5, k=4, name="N=30 P(Z<=5)")
assert_close(mc_p_minor, P_MINOR_EXACT, k=4, name="P(minor) N=1000")
assert_close(mc_Z_major, E_Z_MAJOR, k=4, name="E[Z|major]")
assert_close(mc_p_minor_3, P_MINOR_EXACT_3, k=4, name="P(minor) I0=3")
assert_close(mc_q10, Q10, k=4, name="GW q10")
assert_close(mc_q10_3, Q10**3, k=4, name="GW q10 z0=3")
assert_close(mc_first_inf, P_FIRST_INF, k=4, name="first event infection")
for r0, (pmn, ez, _) in mc_R0.items():
    assert_close(pmn, dp_rows[r0][0], k=4, name=f"P(minor) R0={r0}")
    assert_close(ez, dp_rows[r0][1], k=4, name=f"E[Z|major]/N R0={r0}")
print("all Monte Carlo checks passed (4 SE)")

## 10. 단계 6 — 전염병 보고서

숫자는 모두 위에서 계산한 변수에서 가져온다(하드코딩 없음).

In [ ]:
hdr = "| 질문 | 근사 (분기 / ODE) | 정확 (DP / expm) | MC ± SE |\n|---|---|---|---|\n"
rows6 = [
    ("초기 소멸 확률 P(minor), I0 = 1", f"1/R0 = {1 / R0:.4f}", f"{P_MINOR_EXACT:.6f}", f"{mc_p_minor.mean:.4f} ± {mc_p_minor.se:.4f}"),
    ("초기 소멸 확률, I0 = 3", f"(1/R0)^3 = {(1 / R0)**3:.4f}", f"{P_MINOR_EXACT_3:.6f}", f"{mc_p_minor_3.mean:.4f} ± {mc_p_minor_3.se:.4f}"),
    ("대유행 최종 규모 E[Z | major]", f"N z = {N * Z_EQ:.1f}; ODE {ODE_FINAL:.1f}", f"{E_Z_MAJOR:.3f} (sd {SD_Z_MAJOR:.1f})", f"{mc_Z_major.mean:.1f} ± {mc_Z_major.se:.1f}"),
    ("정점 시각 (major)", f"ODE {T_PEAK_ODE:.3f}; 법칙 평균 {law['mean']:.3f} (sd {law['sd']:.3f})", "—", f"{mc_t_peak.mean:.3f} ± {mc_t_peak.se:.3f} (sd {mc['sd']:.2f})"),
    ("정점 크기 (major)", f"ODE {I_PEAK_ODE:.1f}; 점근 {I_PEAK_ASYMP:.1f}", "—", f"{mc_peak_I.mean:.1f} ± {mc_peak_I.se:.1f}"),
    ("N = 30: P(I_5 = 0), E[T_ext]", "—", f"{P_I5_ZERO:.5f}, {E_T_EXT:.4f}", f"{mc_I5_zero.mean:.4f} ± {mc_I5_zero.se:.4f}, {mc_T_ext.mean:.3f} ± {mc_T_ext.se:.3f}"),
    ("R0 = 1.5: P(minor), E[Z|major]/N", f"{1 / 1.5:.4f}, z = {final_size_z(1.5):.4f}", f"{dp_rows[1.5][0]:.6f}, {dp_rows[1.5][1]:.5f}", f"{mc_R0[1.5][0].mean:.4f} ± {mc_R0[1.5][0].se:.4f}, {mc_R0[1.5][1].mean:.4f} ± {mc_R0[1.5][1].se:.4f}"),
]
tools = ("| 노트북 | 도구 | 이 캡스톤에서의 역할 |\n|---|---|---|\n"
         "| 00b | SE, `mc_proportion`/`mc_estimate` | 모든 판정의 눈금 (4 SE) |\n| 01c | 흡수확률·기본행렬 | 최종 규모 = 흡수 상태, 기대 소멸시간 $(-Q_{TT})^{-1}\\mathbf 1$ |\n"
         "| 01g | 골턴–왓슨, `extinction_probability`, `galton_watson` | 초기 단계의 분기 근사, $1/R_0$ |\n| 02a | 경쟁하는 지수 시계 | 다음 사건이 감염일 확률 $R_0S/(R_0S+N)$ → DP |\n"
         "| 03a | Gillespie, `embedded_chain` | 정확한 경로 표본; 점프 사슬 |\n| 03b | `transition_matrix` = $e^{Qt}$ | $N=30$의 $P(I_t=0)$, $E[I_t]$ |\n| 05d | 마팅게일 극한 $W$ | 정점 시각의 시간 이동 법칙 |")
elapsed = time.perf_counter() - T_START
Markdown(hdr + "\n".join("| " + " | ".join(r) + " |" for r in rows6) +
         f"\n\n**세 줄 결론.** (1) 초기 소멸 확률 $\\approx 1/R_0$ (정확값과의 차이 $\\approx 2/N$). (2) 대유행 규모 $\\approx Nz$ ± {SD_Z_MAJOR:.0f}명 — 이봉 분포. "
         f"(3) 정점 시각 = ODE 정점 − $\\ln W/(\\beta-\\gamma)$, sd ≈ {law['sd']:.2f}는 $N$과 무관.\n\n" + tools +
         f"\n\n총 실행시간 {elapsed:.1f} s (fast={fast()}, N_PATHS={N_PATHS}, N_SMALL={N_SMALL}).")

## 11. 연습문제

### E1 계산

$N = 3$, $I_0 = 1$, $R_0 = 2$인 마르코프 SIR의 최종 규모 분포 $P(Z = 1), P(Z = 2), P(Z = 3)$을 점프 사슬 확률 $p(S) = R_0 S/(R_0 S + N)$으로 손으로 구하고 $E[Z]$를 계산하라. $P(Z = 1)$이 분기 근사 $1/R_0 = 1/2$와 다른 이유를 설명하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$S_0 = 2$: $p(2) = 4/7$, $p(1) = 2/5$. 상태 $(2, 1)$에서 첫 사건이 회복이면 $Z = 1$: $P(Z = 1) = 3/7$. 감염(4/7)이면 $(1, 2)$로 가고, 여기서 감염 없이 회복 두 번($(1, 2) \to (1, 1) \to (1, 0)$)이면 $Z = 2$: 각 단계에서 회복 확률은 $1 - p(1) = 3/5$($I$에 무관!)이므로 $P(Z = 2) = \frac47 \cdot \big(\frac35\big)^2 = \frac{36}{175}$. 나머지 $P(Z = 3) = 1 - \frac37 - \frac{36}{175} = \frac{64}{175}$. $E[Z] = \frac37 + \frac{72}{175} + \frac{192}{175} = \frac{75 + 264}{175} = 1.93714$.

$P(Z = 1) = 3/7 = 0.4286 < 1/2$: 감수성자가 2명뿐이라 첫 사건이 감염일 확률 $p(2) = 4/7$이 분기 근사의 $R_0/(1 + R_0) = 2/3$보다 작다(유효 $R_0 = R_0 S/N = 4/3$). 분기 근사는 $S \approx N$일 때만 맞는다.

```python
pm = sir_final_size_pmf(3, 1, 2.0)              # index s = 0, 1, 2  ->  Z = 3, 2, 1
print(pm[::-1], 3 / 7, 36 / 175, 64 / 175)        # [0.42857 0.20571 0.36571]
print("E[Z] =", pm @ (3 - np.arange(3)))          # 1.93714
```

</details>

### E2 유도 후 시뮬레이션 검증

정리 2의 기하 자손분포와 고정점 $1/R_0$을 직접 유도한 뒤, 감염기간이 고정값 $1/\gamma$이면 자손분포가 Poisson($R_0$)이 되어 소멸확률이 $s = e^{R_0(s - 1)}$의 해임을 보이고, 두 값을 `galton_watson`(각각 기하·Poisson pmf, 15세대, `scale(20000)` 경로)으로 검증하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

기하: 본문의 증명(감염 사건이 회복 사건보다 먼저일 확률 $1 - p = \beta/(\beta + \gamma)$가 매번 독립이므로 회복 전 감염 횟수는 기하). 고정 감염기간 $\tau = 1/\gamma$ 동안 감염 사건은 율 $\beta$의 푸아송 과정이므로($S \approx N$) $K \sim$ Poisson($\beta\tau$) = Poisson($R_0$), pgf $f(s) = e^{R_0(s - 1)}$. 가장 작은 고정점은 $R_0 = 2$에서 0.203188(brentq). 15세대면 $q_{15}$가 극한과 $10^{-4}$ 이내다.

결론: 소멸확률은 감염기간 분포에 민감(0.5 vs 0.203 — 기하 자손은 0을 더 자주 낸다: $P(K = 0) = 1/3$ vs $e^{-2} = 0.135$)하지만, 최종 규모 $z$는 $R_0$만으로 정해진다(E3b).

```python
from scipy.stats import poisson
pois_pmf = poisson.pmf(np.arange(41), R0)
q_pois = brentq(lambda s: s - np.exp(R0 * (s - 1)), 1e-9, 0.999)       # 0.203188
print(q_pois, extinction_probability(pois_pmf))
n_gw = scale(20_000)
ext_geom = mc_proportion(galton_watson(geom_pmf, 15, rng, n_paths=n_gw)[:, -1] == 0)
ext_pois = mc_proportion(galton_watson(pois_pmf, 15, rng, n_paths=n_gw)[:, -1] == 0)
print(ext_geom, gw_extinct_by_generation(geom_pmf, 15))                  # ≈ 0.50
print(ext_pois, gw_extinct_by_generation(pois_pmf, 15))                  # ≈ 0.203
assert_close(ext_geom, gw_extinct_by_generation(geom_pmf, 15), k=4)
assert_close(ext_pois, gw_extinct_by_generation(pois_pmf, 15), k=4)
```

</details>

### E3 가정을 깨보기

(a) $R_0 = 0.8$($\beta = 0.8$)로 두고 $N = 1000$ DP와 Gillespie로 최종 규모 분포를 구하라 — 대유행 봉우리가 사라지는가? $E[Z]$를 분기 과정의 총 자손 평균 $1/(1 - R_0) = 5$와 비교하라. (b) 감염기간을 고정 1일로 바꾼 비마르코프 SIR을 이벤트 리스트로 시뮬레이션(inline)해 $P(\text{minor})$와 $E[Z \mid \text{major}]$를 재라 — 어느 것이 바뀌고 어느 것이 안 바뀌는가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) DP: $E[Z] = 4.6527$, $P(Z \le 10) = 0.9040$, $P(Z = 1) = 0.5558 \approx 1/(1 + R_0) = 0.5556$; 분포는 단봉(모두 소규모)이고 대유행 봉우리는 없다. 분기 과정의 총 자손 평균 $1/(1 - R_0) = 5$보다 작은 것은 감수성 고갈(유효 $R_0$가 $0.8 S/N < 0.8$) 때문이다.

(b) 고정 감염기간: 각 감염자의 회복 시각 = 감염 시각 + 1(힙에 보관). 감염 사건은 현재 $I$에 비례한 율 $\beta SI/N$의 푸아송으로 발생하되 $I$는 다음 회복 시각에서 바뀌므로, 다음 회복 시각까지의 지수 시계를 뽑고 그보다 늦으면 회복 시각으로 점프해 시계를 다시 시작한다(지수의 무기억성 덕에 유효; 02c thinning과 같은 결과). 결과: $P(\text{minor}) \approx 0.20$(Poisson 자손, E2)으로 **바뀌고**, $E[Z \mid \text{major}]/N \approx 0.797$은 **유지**된다 — 최종 규모 방정식은 감염기간의 평균만 본다.

```python
import heapq
pm08 = sir_final_size_pmf(N, 1, 0.8)
print("E[Z] =", pm08 @ Z_vals, " P(Z<=10) =", pm08[Z_vals <= 10].sum(), " P(Z=1) =", pm08[Z_vals == 1][0], 1 / 1.8)
Z08 = sir_gillespie_batch(N, 0.8, GAMMA, 1, N_PATHS, rng)[0]
print(mc_estimate(Z08), " max Z =", Z08.max())                            # ≈ 4.65, 대유행 없음

def sir_fixed_period_path(N, beta, tau, I0, rng):
    S, I, t = N - I0, I0, 0.0
    heap = [tau] * I0
    heapq.heapify(heap)
    while heap:
        t_rec = heap[0]
        a_inf = beta * S * I / N
        t_cand = t + rng.exponential(1.0 / a_inf) if a_inf > 0 else np.inf
        if t_cand < t_rec:                       # 감염이 다음 회복보다 먼저
            t, S, I = t_cand, S - 1, I + 1
            heapq.heappush(heap, t + tau)
        else:                                    # 회복: 시계 재시작 (무기억성)
            t, I = t_rec, I - 1
            heapq.heappop(heap)
    return N - S

Zf = np.array([sir_fixed_period_path(N, BETA, 1.0, 1, rng) for _ in range(scale(20_000, minimum=2_000))])
print(mc_proportion(Zf <= MINOR), "vs Poisson 0.2032;  E[Z|major]/N:", mc_estimate(Zf[Zf > MINOR] / N), "vs z =", Z_EQ)
```

</details>

### 스트레치 S1–S3 (선택)

**S1 ($R_0$ 추정).** 대유행 경로 하나에서 $R_0$을 두 가지로 추정하라: (i) 초기 지수 성장률 $r = \beta - \gamma$($I(t)$가 10~100인 구간의 $\ln I$ 회귀 기울기) → $R_0 = 1 + r/\gamma$; (ii) 최종 규모 $z$에서 $R_0 = -\ln(1 - z)/z$ 역산. 100개 경로에 대한 추정치 분포를 그려 어느 쪽이 정확한지 보라.

<details><summary>풀이 개요</summary>

(ii)가 훨씬 낫다: $z$의 sd 0.029에 델타법을 적용하면 sd ≈ 0.1이고 $z = 0.7968 \to R_0 = -\ln(0.2032)/0.7968 = 2.000$으로 편향이 없다(100경로: 평균 ≈ 2.02, sd ≈ 0.11). (i)는 초기 잡음 때문에 분산이 크고(sd ≈ 0.23) **아래로 편향**된다(평균 ≈ 1.8): $I \in [10, 100]$ 구간에서 이미 $S$가 800대로 줄어 유효 성장률 $\beta S/N - \gamma < 1$이기 때문이다. 구간을 더 앞으로 당기면 편향은 줄고 잡음은 커진다.

```python
est_r, est_z = [], []
while len(est_z) < 100:
    times, Ss, Is = sir_gillespie_path(N, BETA, GAMMA, I0, rng)
    Zp = N - Ss[-1]
    if Zp <= MINOR:
        continue
    k = np.argmax(Is)
    m = (Is[:k] >= 10) & (Is[:k] <= 100)
    est_r.append(1 + np.polyfit(times[:k][m], np.log(Is[:k][m]), 1)[0] / GAMMA)
    z = Zp / N
    est_z.append(-np.log(1 - z) / z)
print(np.mean(est_r), np.std(est_r), np.mean(est_z), np.std(est_z))      # ≈ 1.8, 0.23, 2.02, 0.11
```

</details>

**S2 (백신·집단면역 문턱).** 인구의 비율 $v$가 면역이면 $S_0 = N(1 - v) - 1$. 집단면역 문턱 $1 - 1/R_0 = 0.5$를 DP로 확인하라: $v \in \{0.3, 0.5, 0.6\}$에서 $P(Z > 100)$과 $E[Z]$를 구하고 Gillespie로 $v = 0.5$를 검증하라.

<details><summary>풀이 개요</summary>

DP($S_0 = 699, 499, 399$; 접촉률은 여전히 $\beta SI/N$, $N = 1000$): $P(Z > 100)$ = 0.2717, 0.0277, 0.0005; $E[Z]$ = 98.7, 10.9, 4.35; 유효 $R_0 = R_0(1 - v)$ = 1.4, 1.0, 0.8. 문턱 $v = 0.5$(유효 $R_0 = 1$, 임계)에서 대유행이 거의 사라진다. Gillespie는 `sir_gillespie_batch`의 초기 `S`를 `N - I0` 대신 `S0`으로 바꾼 사본(또는 인자 추가)으로 돌린다.

```python
for v in (0.3, 0.5, 0.6):
    S0v = int(N * (1 - v)) - 1
    pm = sir_final_size_pmf(N, 1, R0, S0=S0v)
    Zv = S0v + 1 - np.arange(S0v + 1)
    print(v, pm[Zv > MINOR].sum(), pm @ Zv)      # 0.2717/98.7, 0.0277/10.9, 0.0005/4.35
```

</details>

**S3 ($N$-스케일링).** 분기 근사의 오차 $P(\text{minor}) - 1/R_0$이 $N$에 어떻게 의존하는지 DP로 $N \in \{100, 300, 1000, 3000\}$에서 구하고(문턱 $0.1N$) $N \times$오차를 표로 만들어 $O(1/N)$임을 확인하라. $E[Z \mid \text{major}]/N \to z$의 수렴도 보라.

<details><summary>풀이 개요</summary>

$N \times (P - 1/2)$: 0.56 ($N = 100$; 문턱 10이라 약간 다름), 2.01, 2.07, 2.02 → 상수 ≈ 2, 즉 오차 ≈ $2/N$. $E[Z \mid \text{major}]/N$: 0.7566, 0.7905, 0.7954, 0.7963 → $z = 0.7968$. $N = 3000$ DP는 약 3초.

```python
for n in (100, 300, 1000, 3000):
    pm = sir_final_size_pmf(n, 1, R0)
    Zn = n - np.arange(n)
    mj = Zn > 0.1 * n
    print(n, n * (pm[~mj].sum() - 1 / R0), (pm[mj] @ Zn[mj]) / pm[mj].sum() / n)
```

</details>

## 12. 정리

1. 마르코프 SIR는 $(S, I)$ 위의 CTMC(율 $\beta SI/N$, $\gamma I$)이고 Gillespie가 정확한 표본을 준다; 점프 사슬의 감염 확률 $R_0 S/(R_0 S + N)$은 $I$에 무관해 최종 규모 분포를 DP로 정확히 계산할 수 있다(작은 $N$에서는 expm·흡수확률로 교차 검증: $N = 30$에서 $10^{-16}$ 일치).
2. 초기 단계는 기하 자손($p = \gamma/(\beta + \gamma)$, 평균 $R_0$)의 분기 과정: 소멸확률 $1/R_0$($I_0 = k$: $(1/R_0)^k$); 유한 $N = 1000$의 정확값 0.502는 $2/N$만큼 크다.
3. 최종 규모는 이봉: 절반은 10명 남짓, 나머지는 $795 \pm 29$(ODE 최종 규모 방정식 $z = 1 - e^{-R_0 z}$, $Nz = 797$). ODE는 두 번째 봉우리만 본다.
4. 정점 시각은 ODE(6.757)를 $-\ln W/(\beta - \gamma)$만큼 옮긴 분포(평균 6.64, sd 1.28)로 흩어진다 — 초기 단계의 마팅게일 극한 $W$가 시계를 정한다(05d). MC sd 1.45는 정점 부근 잡음이 더해진 값.
5. $1/R_0$은 지수 감염기간에 민감(고정 감염기간: 0.203)하지만 $z$는 $R_0$만의 함수 — 모형 가정이 어떤 결론에 영향을 주는지 구분해야 한다.

**trap 카드**
- Q: $R_0 = 2$이면 전염병은 반드시 크게 번지는가?
- A: 아니다. 감염자 1명에서 시작하면 확률 $1/R_0 = 0.5$(정확 0.502)로 10명도 못 채우고 꺼진다 — 분기 과정의 소멸. 대유행이 나면 최종 규모는 거의 확정적($0.797N \pm 29$)이라 분포는 이봉이다. 결정론적 ODE는 '항상 대유행' 쪽만 보여 준다.

**다음 노트북**: 00a — 과정의 끝이다. 닫힌 책으로 00a를 다시 열어 처음 예측을 다시 해 보고 mistakes.md를 통독한 뒤, Season 2(GP 회귀, Lévy 과정, HMM/Kalman, MDP, Girsanov, Fokker–Planck)를 계획한다.

log/progress.md 한 줄 템플릿:
`- [ ] C5 캡스톤 5: 확률적 SIR 전염병 모형 — 날짜: ____ / 예측 적중: __/11 / 단계 1-6 __ / E1 __ E2 __ E3 __ / 메모: ____`